##  Loading the Cleaned CSV

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, avg, stddev, count, when, sum as spark_sum,
    round as spark_round, max as spark_max, min as spark_min
)
from pyspark.sql.window import Window
import time

In [2]:
# ── Start SparkSession ──
spark = SparkSession.builder \
    .appName("LogiScale_Phase2_BigDataProcessing") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "100") \
    .getOrCreate()


In [3]:
# ── Load cleaned CSV from Phase 1 ──
df = spark.read.csv(
    "../dataco_cleaned.csv",
    header=True,
    inferSchema=True
)

In [4]:
print(f"Rows loaded : {df.count():,}")
print(f"Columns     : {len(df.columns)}")
df.printSchema()

Rows loaded : 180,519
Columns     : 32
root
 |-- Type: string (nullable = true)
 |-- ata_days: integer (nullable = true)
 |-- eta_days: integer (nullable = true)
 |-- benefit_per_order: double (nullable = true)
 |-- sales_per_customer: double (nullable = true)
 |-- delivery_status: string (nullable = true)
 |-- late_risk: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- customer_country: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- warehouse_id: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- market: string (nullable = true)
 |-- order_city: string (nullable = true)
 |-- order_country: string (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- units_sold: integer (nullable = true)
 |-- sales_amount: double (nullable = true)
 |-- profit_per_order: double (nullable = true

## ------------------(Core Aggregation)----------------------
 ## GroupBy Route → Average Delay 

In [5]:

print("\n" + "="*50)
print("  Average Delay by Route")
print("="*50)

df_route_delay = df.groupBy("route", "shipping_mode").agg(

    spark_round(avg("delay_days"),   2).alias("avg_delay_days"),
    spark_round(stddev("delay_days"),2).alias("stddev_delay_days"),
    spark_round(avg("ata_days"),     2).alias("avg_actual_days"),
    spark_round(avg("eta_days"),     2).alias("avg_scheduled_days"),

    count("*").alias("total_orders"),

    spark_round(
        count(when(col("delivery_flag") == "Late",    1)) /
        count("*") * 100, 1
    ).alias("late_pct"),

    spark_round(
        count(when(col("delivery_flag") == "On Time", 1)) /
        count("*") * 100, 1
    ).alias("on_time_pct"),

    spark_round(
        count(when(col("delivery_flag") == "Early",   1)) /
        count("*") * 100, 1
    ).alias("early_pct")
)

# Show worst routes first
df_route_delay.orderBy("avg_delay_days", ascending=False).show(15, truncate=False)


  Average Delay by Route
+---------------+-------------+--------------+-----------------+---------------+------------------+------------+--------+-----------+---------+
|route          |shipping_mode|avg_delay_days|stddev_delay_days|avg_actual_days|avg_scheduled_days|total_orders|late_pct|on_time_pct|early_pct|
+---------------+-------------+--------------+-----------------+---------------+------------------+------------+--------+-----------+---------+
|Central Asia   |Second Class |2.21          |1.22             |4.21           |2.0               |117         |90.6    |9.4        |0.0      |
|Central Africa |Second Class |2.13          |1.44             |4.13           |2.0               |297         |82.5    |17.5       |0.0      |
|Eastern Asia   |Second Class |2.12          |1.37             |4.12           |2.0               |1406        |83.8    |16.2       |0.0      |
|Western Europe |Second Class |2.06          |1.42             |4.06           |2.0               |5438       

## GroupBy Warehouse → Inventory & Sales Metrics

In [6]:
print("\n" + "="*50)
print("  Warehouse-level Metrics")
print("="*50)

df_warehouse = df.groupBy("warehouse_id", "market").agg(

    count("*").alias("total_orders"),
    spark_round(spark_sum("units_sold"),    2).alias("total_units_sold"),
    spark_round(spark_sum("sales_amount"),  2).alias("total_sales"),
    spark_round(spark_sum("profit_per_order"), 2).alias("total_profit"),
    spark_round(avg("delay_days"),          2).alias("avg_delay_days"),

    spark_round(
        count(when(col("late_risk") == 1, 1)) /
        count("*") * 100, 1
    ).alias("late_risk_pct")
)

df_warehouse.orderBy("total_orders", ascending=False).show(15, truncate=False)


  Warehouse-level Metrics
+------------+------------+------------+----------------+-----------+------------+--------------+-------------+
|warehouse_id|market      |total_orders|total_units_sold|total_sales|total_profit|avg_delay_days|late_risk_pct|
+------------+------------+------------+----------------+-----------+------------+--------------+-------------+
|Fan Shop    |LATAM       |20009       |32018           |5134797.38 |544466.51   |0.56          |54.7         |
|Fan Shop    |Europe      |18265       |28971           |4740311.57 |510860.74   |0.57          |55.4         |
|Fan Shop    |Pacific Asia|14266       |22498           |3554859.26 |362585.04   |0.55          |54.6         |
|Apparel     |LATAM       |13844       |28445           |2165895.62 |238659.8    |0.55          |53.7         |
|Apparel     |Europe      |13778       |27316           |2324747.9  |272063.22   |0.57          |55.2         |
|Apparel     |Pacific Asia|11103       |21311           |1877122.85 |200975.6

## GroupBy Category/SKU → Demand Summary

In [7]:
print("\n" + "="*50)
print("  Demand by Product Category")
print("="*50)

df_demand = df.groupBy("warehouse_id", "category", "sku_name").agg(

    spark_round(spark_sum("units_sold"),   2).alias("total_units"),
    spark_round(spark_sum("sales_amount"), 2).alias("total_sales"),
    spark_round(avg("units_sold"),         2).alias("avg_units_per_order"),
    count("*").alias("total_orders")
)

df_demand.orderBy("total_units", ascending=False).show(15, truncate=False)


  Demand by Product Category
+------------+--------------------+---------------------------------------------+-----------+-----------+-------------------+------------+
|warehouse_id|category            |sku_name                                     |total_units|total_sales|avg_units_per_order|total_orders|
+------------+--------------------+---------------------------------------------+-----------+-----------+-------------------+------------+
|Apparel     |Cleats              |Perfect Fitness Perfect Rip Deck             |73698      |4421143.14 |3.01               |24515       |
|Golf        |Women's Apparel     |Nike Men's Dri-FIT Victory Golf Polo         |62956      |3147800.0  |2.99               |21035       |
|Fan Shop    |Indoor/Outdoor Games|O'Brien Men's Neoprene Life Vest             |57803      |2888993.91 |3.0                |19298       |
|Footwear    |Cardio Equipment    |Nike Men's Free 5.0+ Running Shoe            |36680      |3667633.2  |3.01               |12169      

## ------------------- Window Function ------------------
##  1: Running Total of Orders per Route

In [8]:
print("\n" + "="*50)
print("  WINDOW 1: Running Total of Orders per Route")
print("="*50)

# Daily order count per route
df_daily_orders = df.groupBy("route", "order_date").agg(
    count("*").alias("daily_orders"),
    spark_round(spark_sum("sales_amount"), 2).alias("daily_sales")
)

# Running total window — cumulative from start to current date
window_running = Window \
    .partitionBy("route") \
    .orderBy("order_date") \
    .rowsBetween(Window.unboundedPreceding, 0)

df_running_total = df_daily_orders \
    .withColumn(
        "running_total_orders",
        spark_sum("daily_orders").over(window_running)
    ) \
    .withColumn(
        "running_total_sales",
        spark_round(spark_sum("daily_sales").over(window_running), 2)
    )

df_running_total.orderBy("route", "order_date").show(20, truncate=False)


  WINDOW 1: Running Total of Orders per Route
+------+-------------------+------------+-----------+--------------------+-------------------+
|route |order_date         |daily_orders|daily_sales|running_total_orders|running_total_sales|
+------+-------------------+------------+-----------+--------------------+-------------------+
|Canada|2016-08-27 02:39:00|3           |919.88     |3                   |919.88             |
|Canada|2016-08-27 04:45:00|2           |179.97     |5                   |1099.85            |
|Canada|2016-08-27 07:54:00|3           |359.98     |8                   |1459.83            |
|Canada|2016-08-27 08:15:00|5           |1209.88    |13                  |2669.71            |
|Canada|2016-08-27 08:36:00|3           |649.91     |16                  |3319.62            |
|Canada|2016-08-27 08:57:00|4           |1014.88    |20                  |4334.5             |
|Canada|2016-08-27 20:52:00|3           |599.97     |23                  |4934.47            |
|Ca

##  2: 7-Day Moving Average of Delay per Route

In [9]:
print("\n" + "="*50)
print("  WINDOW 2: 7-Day Moving Average of Delay")
print("="*50)

# Daily average delay per route
df_daily_delay = df.groupBy("route", "order_date").agg(
    spark_round(avg("delay_days"), 2).alias("avg_daily_delay"),
    count("*").alias("daily_orders")
)

# 7-day moving average window (lookback 6 days + current = 7 days)
window_7d = Window \
    .partitionBy("route") \
    .orderBy("order_date") \
    .rowsBetween(-6, 0)

df_moving_avg_delay = df_daily_delay \
    .withColumn(
        "moving_avg_7d_delay",
        spark_round(avg("avg_daily_delay").over(window_7d), 2)
    )

df_moving_avg_delay.orderBy("route", "order_date").show(20, truncate=False)


  WINDOW 2: 7-Day Moving Average of Delay
+------+-------------------+---------------+------------+-------------------+
|route |order_date         |avg_daily_delay|daily_orders|moving_avg_7d_delay|
+------+-------------------+---------------+------------+-------------------+
|Canada|2016-08-27 02:39:00|1.0            |3           |1.0                |
|Canada|2016-08-27 04:45:00|-2.0           |2           |-0.5               |
|Canada|2016-08-27 07:54:00|1.0            |3           |0.0                |
|Canada|2016-08-27 08:15:00|1.0            |5           |0.25               |
|Canada|2016-08-27 08:36:00|-1.0           |3           |0.0                |
|Canada|2016-08-27 08:57:00|0.0            |4           |0.0                |
|Canada|2016-08-27 20:52:00|-1.0           |3           |-0.14              |
|Canada|2016-08-28 11:35:00|1.0            |5           |-0.14              |
|Canada|2016-08-30 01:04:00|-2.0           |3           |-0.14              |
|Canada|2016-08-30 10

## 3: 4-Week Moving Average for Demand Forecasting

In [10]:
from pyspark.sql.functions import date_trunc

print("\n" + "="*50)
print("  WINDOW 3: 4-Week Moving Avg — Demand Forecast")
print("="*50)

# Weekly units sold per category per warehouse
df_weekly_demand = df \
    .withColumn("week", date_trunc("week", col("order_date"))) \
    .groupBy("warehouse_id", "category", "week") \
    .agg(
        spark_round(spark_sum("units_sold"), 2).alias("weekly_units"),
        spark_round(spark_sum("sales_amount"), 2).alias("weekly_sales")
    )

# 4-week moving average window
window_4w = Window \
    .partitionBy("warehouse_id", "category") \
    .orderBy("week") \
    .rowsBetween(-3, 0)   # 4 weeks lookback

df_demand_forecast = df_weekly_demand \
    .withColumn(
        "moving_avg_4w_units",
        spark_round(avg("weekly_units").over(window_4w), 1)
    ) \
    .withColumn(
        "moving_avg_4w_sales",
        spark_round(avg("weekly_sales").over(window_4w), 2)
    )

df_demand_forecast.orderBy(
    "warehouse_id", "category", "week"
).show(20, truncate=False)


  WINDOW 3: 4-Week Moving Avg — Demand Forecast
+------------+-------------------+-------------------+------------+------------+-------------------+-------------------+
|warehouse_id|category           |week               |weekly_units|weekly_sales|moving_avg_4w_units|moving_avg_4w_sales|
+------------+-------------------+-------------------+------------+------------+-------------------+-------------------+
|Apparel     |Baby               |2017-10-02 00:00:00|82          |4844.56     |82.0               |4844.56            |
|Apparel     |Baby               |2017-12-11 00:00:00|51          |3013.08     |66.5               |3928.82            |
|Apparel     |Baby               |2017-12-18 00:00:00|74          |4371.92     |69.0               |4076.52            |
|Apparel     |Children's Clothing|2017-10-16 00:00:00|305         |108915.5    |305.0              |108915.5           |
|Apparel     |Children's Clothing|2017-10-23 00:00:00|217         |77490.7     |261.0              |9320

##  4: Rank Routes by Worst Delay

In [11]:
from pyspark.sql.functions import rank, dense_rank
from pyspark.sql.window import Window

print("\n" + "="*50)
print("  WINDOW 4: Route Ranking by Avg Delay")
print("="*50)

# Rank all routes — worst delay gets rank 1
window_rank = Window.orderBy(col("avg_delay_days").desc())

df_ranked_routes = df_route_delay \
    .withColumn("delay_rank", dense_rank().over(window_rank))

df_ranked_routes.select(
    "delay_rank", "route", "shipping_mode",
    "avg_delay_days", "late_pct", "total_orders"
).orderBy("delay_rank").show(20, truncate=False)


  WINDOW 4: Route Ranking by Avg Delay
+----------+---------------+-------------+--------------+--------+------------+
|delay_rank|route          |shipping_mode|avg_delay_days|late_pct|total_orders|
+----------+---------------+-------------+--------------+--------+------------+
|1         |Central Asia   |Second Class |2.21          |90.6    |117         |
|2         |Central Africa |Second Class |2.13          |82.5    |297         |
|3         |Eastern Asia   |Second Class |2.12          |83.8    |1406        |
|4         |Western Europe |Second Class |2.06          |81.1    |5438        |
|5         |South Asia     |Second Class |2.04          |79.9    |1552        |
|5         |South of  USA  |Second Class |2.04          |82.4    |829         |
|6         |Caribbean      |Second Class |2.03          |81.9    |1631        |
|6         |Canada         |Second Class |2.03          |77.1    |166         |
|7         |South America  |Second Class |2.0           |78.3    |2827        |


## ------------------ Code Optimization -------------------------

In [12]:
import time
import os
from pyspark.sql.functions import avg

print("\n" + "="*50)
print("  OPTIMIZATION: Reduce Shuffles")
print("="*50)

# ── BEFORE optimization ──
spark.conf.set("spark.sql.shuffle.partitions", "200")

t0 = time.time()
df.groupBy("route").agg(avg("delay_days")).collect()
time_before = round(time.time() - t0, 2)

# ── AFTER optimization ──
optimal_partitions = os.cpu_count() * 2
spark.conf.set("spark.sql.shuffle.partitions", str(optimal_partitions))

# Cache for reuse
df.cache()
df.count()  # trigger cache

t0 = time.time()
df.groupBy("route").agg(avg("delay_days")).collect()
time_after = round(time.time() - t0, 2)

# Safe improvement calculation
if time_before > 0:
    improvement = round((time_before - time_after) / time_before * 100, 1)
else:
    improvement = 0

print(f"\n  Shuffle partitions before : 200")
print(f"  Shuffle partitions after  : {optimal_partitions}")
print(f"  Time before optimization  : {time_before}s")
print(f"  Time after optimization   : {time_after}s")
print(f"  Speed improvement         : {improvement}%")


  OPTIMIZATION: Reduce Shuffles

  Shuffle partitions before : 200
  Shuffle partitions after  : 16
  Time before optimization  : 1.64s
  Time after optimization   : 0.9s
  Speed improvement         : 45.1%


In [13]:
from pyspark.sql.functions import avg, count

print("\n" + "="*50)
print("  SHUFFLE COMPARISON: BAD vs GOOD")
print("="*50)

# ── BAD APPROACH ──
print("\n--- BAD: Two separate groupBy = two shuffles ---")

bad1 = df.groupBy("route").agg(avg("delay_days").alias("avg_delay"))
bad2 = df.groupBy("route").agg(count("*").alias("total_orders"))

print("\nPlan for avg(delay):")
bad1.explain("formatted")

print("\nPlan for count:")
bad2.explain("formatted")


# ── GOOD APPROACH ──
print("\n--- GOOD: One groupBy = one shuffle ---")

good = df.groupBy("route").agg(
    avg("delay_days").alias("avg_delay"),
    count("*").alias("total_orders")
)

print("\nOptimized Plan:")
good.explain("formatted")


  SHUFFLE COMPARISON: BAD vs GOOD

--- BAD: Two separate groupBy = two shuffles ---

Plan for avg(delay):
== Physical Plan ==
AdaptiveSparkPlan (7)
+- HashAggregate (6)
   +- Exchange (5)
      +- HashAggregate (4)
         +- InMemoryTableScan (1)
               +- InMemoryRelation (2)
                     +- Scan csv  (3)


(1) InMemoryTableScan
Output [2]: [route#39, delay_days#47]
Arguments: [route#39, delay_days#47]

(2) InMemoryRelation
Arguments: [Type#17, ata_days#18, eta_days#19, benefit_per_order#20, sales_per_customer#21, delivery_status#22, late_risk#23, category#24, customer_country#25, customer_id#26, customer_segment#27, warehouse_id#28, latitude#29, longitude#30, market#31, order_city#32, order_country#33, order_id#34, order_item_id#35, units_sold#36, sales_amount#37, profit_per_order#38, route#39, order_state#40, order_status#41, ... 7 more fields], StorageLevel(disk, memory, deserialized, 1 replicas)

(3) Scan csv 
Output [32]: [Type#17, ata_days#18, eta_days#19, ben

## Save All Results

In [14]:
import os
import sqlalchemy

print("\n" + "="*50)
print("  SAVING RESULTS TO SQL + CSV")
print("="*50)

# ── Get project root folder ──
notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir)

# ── outputs folder in project root ──
output_dir = os.path.join(project_root, "outputs")

# Create outputs folder if not exists
os.makedirs(output_dir, exist_ok=True)

print(f"  Notebook running from : {notebook_dir}")
print(f"  Outputs saving to     : {output_dir}")

# ── SQLite database path ──
engine = sqlalchemy.create_engine(
    f"sqlite:///{output_dir}/logiscale.db"
)

# ── Function to save tables ──
def save_table(spark_df, name, limit_rows=None):

    print(f"\n  Saving: {name}")

    if limit_rows:
        spark_df = spark_df.limit(limit_rows)

    # Convert Spark → Pandas
    pdf = spark_df.toPandas()

    # Save CSV
    csv_path = os.path.join(output_dir, f"{name}.csv")
    pdf.to_csv(csv_path, index=False)

    # Save SQL table
    pdf.to_sql(name, engine, if_exists="replace", index=False)

    print(f"  [SAVED] {name} → {len(pdf):,} rows")
    print(f"  Location: {csv_path}")

# ── Save all tables ──
save_table(df_route_delay,      "delay_summary")
save_table(df_warehouse,        "warehouse_summary")
save_table(df_demand,           "demand_summary")
save_table(df_running_total,    "running_totals")
save_table(df_moving_avg_delay, "moving_avg_delay")
save_table(df_demand_forecast,  "demand_forecast")
save_table(df_ranked_routes,    "ranked_routes")

print("\n" + "="*50)
print("  ALL FILES SAVED SUCCESSFULLY")
print(f"  Output Folder: {output_dir}")
print("="*50)


  SAVING RESULTS TO SQL + CSV
  Notebook running from : C:\Users\Anubrata Parida\MyPython\Projects\LogiScale_BigData\notebooks
  Outputs saving to     : C:\Users\Anubrata Parida\MyPython\Projects\LogiScale_BigData\outputs

  Saving: delay_summary
  [SAVED] delay_summary → 92 rows
  Location: C:\Users\Anubrata Parida\MyPython\Projects\LogiScale_BigData\outputs\delay_summary.csv

  Saving: warehouse_summary
  [SAVED] warehouse_summary → 38 rows
  Location: C:\Users\Anubrata Parida\MyPython\Projects\LogiScale_BigData\outputs\warehouse_summary.csv

  Saving: demand_summary
  [SAVED] demand_summary → 118 rows
  Location: C:\Users\Anubrata Parida\MyPython\Projects\LogiScale_BigData\outputs\demand_summary.csv

  Saving: running_totals
  [SAVED] running_totals → 65,752 rows
  Location: C:\Users\Anubrata Parida\MyPython\Projects\LogiScale_BigData\outputs\running_totals.csv

  Saving: moving_avg_delay
  [SAVED] moving_avg_delay → 65,752 rows
  Location: C:\Users\Anubrata Parida\MyPython\Project